## Creation of a Temporary table for all the regions in Beijing via Database Query

## Import Libraries

Load pandas for data manipulation and SQLAlchemy for connecting to the MySQL database.

In [2]:
import pandas as pd
from sqlalchemy import create_engine

## Connect to DATABASE

Create and test a connection to the `air_quality` database.

In [3]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")
except Exception as error:
    print(f"Failed to connect to air_quality: {error}")

Connected to air_quality!


## Combine Station Tables

Load each station table, print its row count, and combine all station data into `df_combined`. The combined data is also exported to `regions.csv`.

In [4]:
table_names = [
    "aotizhongxin", "changping", "dingling", "dongsi",
    "guanyuan", "gucheng", "huairou", "nongzhanguan",
    "shunyi", "tiantan", "wanliu", "wanshouxigong"
]

dataframes = []
source_row_counts = {}

for table_name in table_names:
    dataframe = pd.read_sql_query(
        f"SELECT * FROM `{table_name}`",
        engine
    )
    dataframes.append(dataframe)
    source_row_counts[table_name] = len(dataframe)
    print(f"{table_name}: {len(dataframe):,} rows")

df_combined = pd.concat(dataframes, ignore_index=True)

df_combined_raw = df_combined.copy(deep=True)


source_total_rows = sum(source_row_counts.values())
combined_total_rows = len(df_combined)

print(f"\nRows from all source tables: {source_total_rows:,}")
print(f"Rows in df_combined: {combined_total_rows:,}")
print(f"Row counts match: {source_total_rows == combined_total_rows}")

aotizhongxin: 35,064 rows
changping: 35,064 rows
dingling: 35,064 rows
dongsi: 35,064 rows
guanyuan: 35,064 rows
gucheng: 35,064 rows
huairou: 35,064 rows
nongzhanguan: 35,064 rows
shunyi: 35,064 rows
tiantan: 35,064 rows
wanliu: 35,064 rows
wanshouxigong: 35,064 rows

Rows from all source tables: 420,768
Rows in df_combined: 420,768
Row counts match: True


## Check for All Stations

Compare the stations in `df_combined` with the expected list and report missing or unexpected stations.

In [5]:
expected_stations = {
    "aotizhongxin", "changping", "dingling", "dongsi",
    "guanyuan", "gucheng", "huairou", "nongzhanguan",
    "shunyi", "tiantan", "wanliu", "wanshouxigong"
}

if "station" not in df_combined.columns:
    raise KeyError("The 'station' column was not found in df_combined.")

present_stations = set(df_combined["station"].dropna().str.lower().unique())
missing_stations = expected_stations - present_stations
unexpected_stations = present_stations - expected_stations

print(f"Expected stations: {len(expected_stations)}")
print(f"Stations found: {len(present_stations)}")
print(f"Missing stations: {sorted(missing_stations) or 'None'}")
print(f"Unexpected stations: {sorted(unexpected_stations) or 'None'}")

if not missing_stations and not unexpected_stations:
    print("All expected stations are present.")
else:
    print("The station list does not exactly match the expected stations.")

Expected stations: 12
Stations found: 12
Missing stations: None
Unexpected stations: None
All expected stations are present.


## Check for Duplicates and Null Values
Review duplicate rows and nulls for each column.

In [6]:
required_id_cols = ["No", "year", "month", "day", "station"]
measurement_cols = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM"
]

if not set(required_id_cols).issubset(df_combined.columns):
    missing_required = set(required_id_cols) - set(df_combined.columns)
    raise KeyError(f"Missing required columns: {sorted(missing_required)}")

duplicate_rows = df_combined.duplicated().sum()
print(f"Duplicate rows before cleaning: {duplicate_rows}")

required_nulls = {
    col: int(df_combined[col].isna().sum())
    for col in required_id_cols
}

print("\nNull counts in required identification columns:")
for col, count in required_nulls.items():
    print(f"  {col}: {count}")

all_null_counts = df_combined.isna().sum()
null_columns = all_null_counts[all_null_counts > 0]

print("\nNull counts in all columns with null values:")
if null_columns.empty:
    print("  No null values found in any column.")
else:
    for col, count in null_columns.items():
        print(f"  {col}: {count}")

hour_null_count = int(df_combined["hour"].isna().sum()) if "hour" in df_combined.columns else 0
print(f"\nNull hour count: {hour_null_count}")

if "hour" in df_combined.columns:
    zero_hour_groups = set(
        df_combined.loc[df_combined["hour"] == 0, ["year", "month", "day", "station"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )

    null_hour_duplicates = (
        df_combined.loc[df_combined["hour"].isna()]
        .apply(
            lambda row: (
                row["year"], row["month"], row["day"], row["station"]
            ) in zero_hour_groups,
            axis=1
        )
        .sum()
    )

    print(f"Null-hour rows that duplicate an existing 0-hour entry: {null_hour_duplicates}")

rain_null_count = int(df_combined["RAIN"].isna().sum()) if "RAIN" in df_combined.columns else 0
print(f"Null RAIN count before fill: {rain_null_count}")

if measurement_cols:
    missing_ratio = df_combined[measurement_cols].isna().mean(axis=1)
    rows_above_threshold = int((missing_ratio > 0.5).sum())
    print(f"Rows with >50% missing measurement values: {rows_above_threshold}")
else:
    print("No measurement columns available for missingness threshold check.")

Duplicate rows before cleaning: 0

Null counts in required identification columns:
  No: 0
  year: 0
  month: 0
  day: 0
  station: 0

Null counts in all columns with null values:
  PM2.5: 8739
  PM10: 6449
  SO2: 9021
  NO2: 12116
  CO: 20701
  O3: 13277
  TEMP: 398
  PRES: 393
  DEWP: 403
  RAIN: 390
  wd: 1822
  WSPM: 318

Null hour count: 0
Null-hour rows that duplicate an existing 0-hour entry: 0.0
Null RAIN count before fill: 390
Rows with >50% missing measurement values: 26


## Remove Unacceptable Nulls and Duplicates
Drop invalid rows and fill valid missing values

In [ ]:
df_cleaned = df_combined_raw.copy(deep=True)
df_cleaned["interpolated_columns"] = [[] for _ in range(len(df_cleaned))]

required_id_cols = ["No", "year", "month", "day", "station"]
measurement_cols = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM"
]
pollutant_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
meteorological_cols = ["TEMP", "PRES", "DEWP", "WSPM"]

if not set(required_id_cols).issubset(df_cleaned.columns):
    missing_required = set(required_id_cols) - set(df_cleaned.columns)
    raise KeyError(f"Missing required columns: {sorted(missing_required)}")

row_count_before_step1 = len(df_cleaned)
missing_ratio = df_cleaned[measurement_cols].isna().mean(axis=1)
df_cleaned = df_cleaned.loc[missing_ratio <= 0.5].copy()
row_count_after_step1 = len(df_cleaned)
rows_dropped_by_missing_ratio = int((missing_ratio > 0.5).sum())
print(f"Rows before >50% missing removal: {row_count_before_step1:,}")
print(f"Rows after >50% missing removal: {row_count_after_step1:,}")
print(f"Rows dropped by >50% missing measurement rule: {rows_dropped_by_missing_ratio:,}")

df_cleaned = df_cleaned.sort_values(
    ["station", "year", "month", "day", "hour"],
    kind="mergesort"
    ).reset_index(drop=True)

for col in pollutant_cols:
    if col not in df_cleaned.columns:
        continue
    missing_before = df_cleaned[col].isna().copy()
    df_cleaned[col] = (
        df_cleaned.groupby("station", group_keys=False)[col]
        .transform(
            lambda s: s.interpolate(method="linear", limit=3, limit_direction="both")
        )
    )
    interpolated_mask = missing_before & df_cleaned[col].notna()
    df_cleaned.loc[interpolated_mask, "interpolated_columns"] = (
        df_cleaned.loc[interpolated_mask, "interpolated_columns"]
        .apply(lambda names: names + [col])
    )

for col in meteorological_cols:
    if col not in df_cleaned.columns:
        continue
    missing_before = df_cleaned[col].isna().copy()
    df_cleaned[col] = (
        df_cleaned.groupby("station", group_keys=False)[col]
        .transform(
            lambda s: s.interpolate(method="linear", limit=3, limit_direction="both")
        )
    )
    interpolated_mask = missing_before & df_cleaned[col].notna()
    df_cleaned.loc[interpolated_mask, "interpolated_columns"] = (
        df_cleaned.loc[interpolated_mask, "interpolated_columns"]
        .apply(lambda names: names + [col])
    )

if "RAIN" in df_cleaned.columns:
    rain_non_null = df_cleaned["RAIN"].dropna()
    rain_zero_share = (rain_non_null == 0).mean() if len(rain_non_null) > 0 else 0.0
    print(f"RAIN zero share among non-null values: {rain_zero_share:.2%}")

    if rain_zero_share >= 0.75:
        rain_missing_before = df_cleaned["RAIN"].isna().copy()
        df_cleaned.loc[rain_missing_before, "RAIN"] = 0
        print("RAIN nulls filled with 0 because the series is predominantly dry/zero.")
    else:
        print("RAIN is not predominantly zero; nulls were left for review instead of interpolating.")

if "wd" in df_cleaned.columns:
    wd_missing_before = df_cleaned["wd"].isna().copy()
    df_cleaned["wd"] = (
        df_cleaned.groupby("station", group_keys=False)["wd"]
        .transform(lambda s: s.ffill(limit=3).bfill(limit=3))
    )
    interpolated_mask = wd_missing_before & df_cleaned["wd"].notna()
    df_cleaned.loc[interpolated_mask, "interpolated_columns"] = (
        df_cleaned.loc[interpolated_mask, "interpolated_columns"]
        .apply(lambda names: names + ["wd"])
    )
    remaining_wd_nulls = int(df_cleaned["wd"].isna().sum())

all_pollutant_missing_after = df_cleaned[pollutant_cols].isna().all(axis=1)
rows_dropped_by_all_pollutants_missing = int(all_pollutant_missing_after.sum())

df_cleaned = df_cleaned.loc[~all_pollutant_missing_after].copy()
print(f"Rows dropped because all six pollutant columns are still missing after interpolation: {rows_dropped_by_all_pollutants_missing:,}")

remaining_null_counts = df_cleaned.isna().sum().sort_values(ascending=False)
remaining_null_counts = remaining_null_counts[remaining_null_counts > 0]

if remaining_null_counts.empty:
    print("  No nulls remain in the cleaned dataset.")
else:
    print("\nRemaining null counts per column after interpolation/fill:")
    print(remaining_null_counts)

rows_with_nulls = int(df_cleaned.isna().any(axis=1).sum())
print(f"\nRows still containing nulls after all steps: {rows_with_nulls:,}")

nulls_per_row = df_cleaned[measurement_cols].isna().sum(axis=1)
for count in range(nulls_per_row.max() + 1):
    rows = (nulls_per_row == count).sum()
    print(f"Rows with exactly {count} null(s): {rows:,}")

six_null_rows = df_cleaned[nulls_per_row == 6].copy()
if not six_null_rows.empty:
    missing_pattern = (
        six_null_rows[measurement_cols]
        .isna()
        .astype(int)
        .astype(str)
        .agg("".join, axis=1)
    )
    print("\nMissing-value patterns among rows with exactly 6 nulls after cleaning:")
    print(missing_pattern.value_counts())
else:
    print("\nNo rows with exactly 6 missing measurement values remain after cleaning.")

df_cleaned["interpolated_columns"] = df_cleaned["interpolated_columns"].apply(
    lambda names: ", ".join(names) if names else "None"
 )

df_combined = df_cleaned.copy(deep=True)

display(df_combined[["station", "year", "month", "day", "hour", "interpolated_columns"]].head())


Rows before >50% missing removal: 420,768
Rows after >50% missing removal: 420,742
Rows dropped by >50% missing measurement rule: 26
RAIN zero share among non-null values: 96.07%
RAIN nulls filled with 0 because the series is predominantly dry/zero.
Rows dropped because all six pollutant columns are still missing after interpolation: 1,989

Remaining null counts per column after interpolation/fill:
CO       9996
NO2      3533
O3       3436
SO2      1901
PM2.5    1359
PM10      299
wd          8
TEMP        1
DEWP        1
PRES        1
dtype: int64

Rows still containing nulls after all steps: 17,212
Rows with exactly 0 null(s): 401,541
Rows with exactly 1 null(s): 15,387
Rows with exactly 2 null(s): 1,098
Rows with exactly 3 null(s): 88
Rows with exactly 4 null(s): 507
Rows with exactly 5 null(s): 132

No rows with exactly 6 missing measurement values remain after cleaning.


,station,year,month,day,hour,interpolated_columns
0,Aotizhongxin,2013,3,1,0,None
1,Aotizhongxin,2013,3,1,1,None
2,Aotizhongxin,2013,3,1,2,None
3,Aotizhongxin,2013,3,1,3,None
4,Aotizhongxin,2013,3,1,4,None


## Final DataFrame Verification

In [ ]:
verification_display = df_combined.copy()

print(f"Before Cleaning Rows: {len(df_combined_raw):,}")
print(f"Final DataFrame rows: {len(df_combined):,}")
print(f"Final DataFrame columns: {len(df_combined.columns)}")
print("\nFinal DataFrame Verification Table:")
display(verification_display)


Before Cleaning Rows: 420,768
Final DataFrame rows: 418,753
Final DataFrame columns: 20

Final DataFrame Verification Table:


,station,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,interpolated_columns
0,Aotizhongxin,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,None
1,Aotizhongxin,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,None
2,Aotizhongxin,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,None
3,Aotizhongxin,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,None
4,Aotizhongxin,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,None
5,Aotizhongxin,2013,3,1,5,5.0,5.0,18.0,18.0,400.0,66.0,-2.2,1025.6,-19.6,0.0,N,3.7,None
6,Aotizhongxin,2013,3,1,6,3.0,3.0,18.0,32.0,500.0,50.0,-2.6,1026.5,-19.1,0.0,NNE,2.5,None
7,Aotizhongxin,2013,3,1,7,3.0,6.0,19.0,41.0,500.0,43.0,-1.6,1027.4,-19.1,0.0,NNW,3.8,None
8,Aotizhongxin,2013,3,1,8,3.0,6.0,16.0,43.0,500.0,45.0,0.1,1028.3,-19.2,0.0,NNW,4.1,None
9,Aotizhongxin,2013,3,1,9,3.0,8.0,12.0,28.0,400.0,59.0,1.2,1028.5,-19.3,0.0,N,2.6,None


## Load Combined Data into DATABASE

Store `df_combined` in the `air_quality` database 

In [8]:
df_combined.to_sql(
    "all_regions",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

loaded_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM all_regions",
    engine
)

print(f"Rows in final DataFrame: {len(df_combined):,}")
print(f"Rows in MySQL all_regions table: {loaded_count['total_rows'].iloc[0]:,}")
print(
    f"Load row counts match: "
    f"{loaded_count['total_rows'].iloc[0] == len(df_combined)}"
)


Rows in final DataFrame: 415,791
Rows in MySQL all_regions table: 415,791
Load row counts match: True


## Final SQL Analysis

### Question: Which Beijing monitoring station has the highest average PM2.5 concentration?

PM2.5 is fine particulate matter that can be inhaled into the lungs. It is an important indicator of air pollution, so this analysis identifies which monitoring station has the highest average PM2.5 concentration.

In [9]:
query = """
SELECT
    station,
    AVG(`PM2.5`) AS average_pm25
FROM all_regions
GROUP BY station
ORDER BY average_pm25 DESC;
"""

result = pd.read_sql_query(query, engine)

top_5 = result.head(5)
display(top_5)

top_station = result.iloc[0]

print(
    f"\nTop 1 Station: {top_station['station']}"
    f" with an average PM2.5 concentration of "
    f"{top_station['average_pm25']:.2f}"
)

,station,average_pm25
0,Dongsi,86.194297
1,Wanshouxigong,85.024136
2,Nongzhanguan,84.838483
3,Gucheng,83.852089
4,Wanliu,83.374716



Top 1 Station: Dongsi with an average PM2.5 concentration of 86.19
